### DOE analysis on /theta (transconjugant rate score)

In [5]:
import pandas as pd 
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from numpy.random import default_rng
import os
import arviz as az

# Import functions
from cmdstanpy import CmdStanModel
from tensorflow_probability.substrates import numpy as tfp
tfd = tfp.distributions


import cmdstanpy
import os
from pathlib import Path

import os, sys
from pathlib import Path
def set_project_root():
        
    # Find and navigate to Progetto_Bayesian_Statistics
    current = Path.cwd()

    # Check if we're already there
    if current.name == 'Progetto_Bayesian_Statistics':
        print(f"Already in project root: {current}")
    else:
        # Search up to parent directories
        for parent in [current] + list(current.parents):
            if parent.name == 'Progetto_Bayesian_Statistics':
                os.chdir(parent)
                print(f"Changed to: {parent}")
                break
        else:
            raise FileNotFoundError("Progetto_Bayesian_Statistics folder not found")

    # Verify
    assert Path('src').exists(), "src/ folder not found"
    print(f"Current directory: {os.getcwd()}")
    print(f"Contents: {os.listdir()}")


    # Create ./stan folder if does not exists
    STAN_PATH = "./src/stan_models/"
    print(cmdstanpy.cmdstan_path())
    return STAN_PATH

STAN_PATH = set_project_root()
print("cwd:", os.getcwd())
print("first sys.path entries:")
for p in sys.path[:5]:
    print("  ", p)

print("project root guess:", Path().resolve())
print("exists ./src ?", (Path().resolve() / "src").exists())

from src.utils.utils import *

Already in project root: /Users/sararedaelli/Desktop/Progetto_Bayesian_Statistics
Current directory: /Users/sararedaelli/Desktop/Progetto_Bayesian_Statistics
Contents: ['materials', '.DS_Store', 'requirements.txt', 'README.md', 'results', '.gitignore', 'requirements.bk.txt', '.git', 'data', 'src']
/Users/sararedaelli/.cmdstan/cmdstan-2.37.0
cwd: /Users/sararedaelli/Desktop/Progetto_Bayesian_Statistics
first sys.path entries:
   /Library/Frameworks/Python.framework/Versions/3.13/lib/python313.zip
   /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13
   /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload
   
   /Users/sararedaelli/Library/Python/3.13/lib/python/site-packages
project root guess: /Users/sararedaelli/Desktop/Progetto_Bayesian_Statistics
exists ./src ? True


In [2]:
data, data_donors = transform_data()
glm_data, X_D = get_glm_data(data, data_donors)

[1 2 3 4 5 6 7 8]
[1 2 3 4 5 6 7 8]


/Users/sararedaelli/Desktop/Progetto_Bayesian_Statistics/src/utils/utils.py:70: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Control'] = data['Control'].replace({'Yes': 1, 'No': 0})


In [3]:
model_names = [("ab_baseline_simple_poi.stan", {}), ("ab_lasso.stan", {}), ("ab_horseshoe.stan", {}), ("ab_regularized_horseshoe.stan", {"p0": 2}), ("ab_r2d2.stan", {"R2_mean": 0.5,"R2_prec": 2.0, "cons_D2": 0.5})]

models = {}
for key in model_names:
    glm = CmdStanModel(stan_file=f"{STAN_PATH}/{key[0]}")
    for x in X_D:
        models[(key[0], tuple(key[1].items()), x)] = {"model": glm, "X": X_D[x]}
        new_data = glm_data.copy()
        new_data.update({"X": X_D[x], "p": X_D[x].shape[1]})
        new_data.update(key[1])
        models[(key[0], tuple(key[1].items()), x)]["data"] = new_data

In [4]:
import os
for key in models:
    filename = f"{'_'.join(str(k) for k in key)}.nc"
    models[key]["az"] = az.from_netcdf(f"./results/ab/{filename}")

##### Reconstruct the posterior mean rate per observation

In [10]:
def get_posterior_rate_means(idata, X, idx_experiment, idx_experiment_replica):
    # ---- your original data used to fit the model ----
    exp_idx = np.asarray(idx_experiment)
    rep_idx = np.asarray(idx_experiment_replica)

    # ---- posterior draws ----
    post = idata.posterior
    beta = post["beta"]                      # dims: (chain, draw, p)
    alpha = post["alpha"]                    # dims: (chain, draw, I)
    b_re  = post["beta_random"]              # dims: (chain, draw, I, J)

    # stack chain+draw -> sample dimension for easier math
    beta_s  = beta.stack(sample=("chain","draw")).transpose("sample", ...)
    alpha_s = alpha.stack(sample=("chain","draw")).transpose("sample", ...)
    b_re_s  = b_re.stack(sample=("chain","draw")).transpose("sample", ...)

    # linear predictor per observation i: eta = x_i beta + random intercept
    # X @ beta: (N,) for each sample -> we do matrix multiply with broadcasting
    X_da = xr.DataArray(X, dims=("obs","p"))

    eta = (X_da @ beta_s)  # dims: (obs, sample)
    eta = eta.transpose("sample","obs")

    # add random intercept by indexing (exp,rep) for each obs
    b_obs = b_re_s.isel(I=xr.DataArray(exp_idx, dims="obs"),
                        J=xr.DataArray(rep_idx, dims="obs"))  # dims: (sample, obs)

    # multiply by alpha[exp] and exp(...)
    #alpha_obs = alpha_s.isel(I=xr.DataArray(exp_idx, dims="obs"))  # (sample, obs)

    #mu = alpha_obs * xr.ufuncs.exp(eta + b_obs)  # (sample, obs)
    mu = xr.ufuncs.exp(eta + b_obs)  # (sample, obs)

    # include rho offset if you want the actual Poisson mean
    #rho_da = xr.DataArray(rho_trasc, dims="obs")
    #lam = mu * rho_da  # (sample, obs)

    #mean rate per observation
    rate = mu.mean(dim="sample")  # (obs,)

    return rate

In [12]:
#apply function to the simple model with X simple
get_posterior_rate_means(models[("ab_baseline_simple_poi.stan", tuple({}.items()), "simple")]["az"], 
                         models[("ab_baseline_simple_poi.stan", tuple({}.items()), "simple")]["X"], glm_data["idx_experiment"], glm_data["idx_experiment_replica"])

KeyError: ('ab_baseline_simple_poi.stan', (), 'simple')